In [1]:
#Ideea: un model doar-linie prezice rezistenta unui izolat doar din „familia“ lui (ST sau cluster HC), fara nicio gena. Il evaluezi sub impartire aleatorie si temporala si masori cat de departe e testul de antrenare.

#incarcare
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score

df = pd.read_csv("../data/processed/cipro_norvegia.tsv", sep="\t", dtype=str)
df["collection_year"] = df["collection_year"].astype(float).astype(int)
df["y"] = df["y"].astype(int)
print(df.shape, "prevalenta:", round(df["y"].mean(), 3))

(3148, 14) prevalenta: 0.098


In [2]:
#nivelurile de linie (HC0 = foarte fin, HC100 = larg)
NIVELURI = [0, 2, 5, 10, 20, 50, 100]
MODELE = {"ST": "mlst", **{f"HC{h}": f"cgmlst_hc{h}" for h in NIVELURI}}

In [3]:
#cele doua impartiri (cea aleatorie are aceleasi dimensiuni ca cea temporala, ca sa fie comparabile)
def split_temporal(df, an_taiere=2012):
    return df[df["collection_year"] <= an_taiere], df[df["collection_year"] > an_taiere]

def split_aleatoriu(df, n_train, n_test, seed):
    return train_test_split(df, train_size=n_train, test_size=n_test,
                            stratify=df["y"], random_state=seed)

In [4]:
#modelul doar-linie si distanta
def model_linie(train, test, col, alpha=1.0):
    """Probabilitate = proportia de rezistente din antrenare in acelasi cluster/ST.
    Se trage spre prevalenta globala (alpha), iar un cluster nevazut primeste prevalenta."""
    p0 = train["y"].mean()
    stat = train.groupby(col)["y"].agg(["sum", "count"])
    s = test[col].map(stat["sum"]).fillna(0)
    n = test[col].map(stat["count"]).fillna(0)
    return ((s + alpha * p0) / (n + alpha)).values

def distanta_minima(train, test):
    """Cel mai fin nivel HC la care izolatul de test are un 'var' in antrenare (inf = niciunul)."""
    d = pd.Series(np.inf, index=test.index)
    for h in sorted(NIVELURI, reverse=True):      # de la larg la fin, ultimul scris e cel mai fin
        col = f"cgmlst_hc{h}"
        d[test[col].isin(set(train[col]))] = h
    return d

In [5]:
#evaluarea
def evalueaza(train, test):
    """AUROC si castigul AUPRC fata de prevalenta din test (AUPRC al unui model fara informatie)."""
    prev_test = test["y"].mean()
    rez = {}
    for nume, col in MODELE.items():
        scor = model_linie(train, test, col)
        rez[nume] = (roc_auc_score(test["y"], scor),
                     average_precision_score(test["y"], scor) - prev_test)
    return pd.DataFrame(rez, index=["AUROC", "AUPRC_castig"]).T